In [9]:
import pandas as pd
import numpy as np
import math

#Row-column logic. Ex: Matrix[i][j] refers to row i and column j (indices)

In [10]:
df = pd.read_csv("wdbc.csv")

x = df.iloc[:, 2:]
y = df.iloc[:, 1:2]

rows, cols = x.shape

# Shuffle and split
shuffle_idx = np.random.permutation(rows)
train_size = int(0.8 * rows)

train_idx = shuffle_idx[:train_size]
test_idx = shuffle_idx[train_size:]

x_train = x.iloc[train_idx].values
x_test = x.iloc[test_idx].values
y_train = y.iloc[train_idx, 0].values
y_test = y.iloc[test_idx, 0].values

print(f"Training samples: {len(x_train)}, Test samples: {len(x_test)}")


dimension = x_train.shape
dimension_test = x_test.shape

print(x_train)
print(dimension)
print(y_test.shape)



Training samples: 454, Test samples: 114
[[1.276e+01 1.884e+01 8.187e+01 ... 8.312e-02 2.744e-01 7.238e-02]
 [1.246e+01 1.989e+01 8.043e+01 ... 7.625e-02 2.685e-01 7.764e-02]
 [8.888e+00 1.464e+01 5.879e+01 ... 4.786e-02 2.254e-01 1.084e-01]
 ...
 [1.450e+01 1.089e+01 9.428e+01 ... 1.221e-01 2.889e-01 8.006e-02]
 [1.571e+01 1.393e+01 1.020e+02 ... 1.374e-01 2.723e-01 7.071e-02]
 [2.573e+01 1.746e+01 1.742e+02 ... 2.756e-01 3.690e-01 8.815e-02]]
(454, 30)
(114,)


In [11]:
def compute_phi_malignant():
    count = 0
    for i in range(train_size):
        if y_train[i] == 'M':
            count += 1

    return count/train_size

phi = compute_phi_malignant()
print(phi)
    

0.3700440528634361


In [12]:
def mu_zero():
    init_arr = np.zeros(len(x_train[0]))
    for i in range(train_size):
        if y_train[i] != "M":
            for j in range(len(init_arr)):
                init_arr[j] += x_train[i][j]
    
    for k in range(len(init_arr)):
        init_arr[k] /= ((1-phi) * train_size)

    return init_arr.tolist()

mu_zero_arr = mu_zero()

In [13]:
def mu_one():
    init_arr = np.zeros(len(x_train[0]))
    for i in range(train_size):
        if y_train[i] == "M":
            for j in range(len(init_arr)):
                init_arr[j] += x_train[i][j]
    
    for k in range(len(init_arr)):
        init_arr[k] /= (phi* train_size)

    return init_arr.tolist()

mu_one_arr = mu_one()

In [14]:
#This function calculates the matrix derived from the product of a vector and its transpose
def matrix_product_from_vector_and_transpose(dimension, vector):
    res = [[0] * dimension for _ in range(dimension)]
    for i in range(dimension):
        for j in range(dimension):

            #Transpose product
            res [i][j] = vector[i] * vector[j]
    
    return res

#This calculates the resultant vector (v1 - v2)
def calculate_vector(v1, v2):
    res = v1.copy()
    for i in range(len(res)):
        res[i] -= v2[i]
    return res

#Calculates the covariance matrix using formulae
def calculate_covariance_matrix(dimension):
    res = [[0] * dimension for _ in range(dimension)]

    #Goes through each training example
    for i in range(train_size):
        #Checks type of training exmaple
        v2 = []
        if y_train[i] == 'M':
            v2 = mu_one_arr
        else:
            v2 = mu_zero_arr
        #Calculates resultant vector based on which mu it is(depending upon the type of training example)
        vector = calculate_vector(x_train[i], v2)
        #Calculates matrix product with transpose
        mat = matrix_product_from_vector_and_transpose(dimension, vector)
        for j in range(dimension):
            for k in range(dimension):
                #Division by training size
                res[j][k] += (mat[j][k])/train_size
    
    return res

In [15]:
def LU_decomposition(matrix, dimension):

    #Rewriting matrix as the product of an upper triangle and a lower triangle matrix
    #Using the doolittle method of initialization
    lower = [[0] * dimension for _ in range(dimension)]
    upper = [[0] * dimension for _ in range(dimension)]

    # Initialize first row of U and first column of L
    for i in range(dimension):
        lower[i][i] = 1
        upper[0][i] = matrix[0][i]
        lower[i][0] = matrix[i][0] / upper[0][0]

    for j in range(1, dimension):

        # Calculating rows of upper matrix
        for k in range(j + 1):
            sum_u = 0
            for s in range(k):
                sum_u += lower[k][s] * upper[s][j]
            upper[k][j] = matrix[k][j] - sum_u

        # Calculating columns of lower matrix
        for k in range(j + 1, dimension):
            sum_l = 0
            for s in range(j):
                sum_l += lower[k][s] * upper[s][j]
            lower[k][j] = (1/upper[j][j])*(matrix[k][j] - sum_l)

    
    return lower, upper

def calculate_determinant(upper, dimension):
    # Determinant is product of U's diagonal
    det = 1
    for i in range(dimension):
        det *= upper[i][i]

    return det


def matrix_inverse(lower, upper, dimension):
    Y_res = [[0] * dimension for _ in range(dimension)]
    X_res = [[0] * dimension for _ in range(dimension)]
    identity = [[0] * dimension for _ in range (dimension)]
    for m in range(dimension):
        identity [m][m] = 1

    
    for i in range(dimension):
        for j in range(dimension):
            sum_Y = 0
            for k in range(i):
                sum_Y += lower[i][k] * Y_res[k][j]
            Y_res[i][j] = identity[i][j] - sum_Y
    

    for i in range(dimension - 1, -1, -1):  # iterate bottom to top
        for j in range(dimension):
            sum_X = 0
            for k in range(i + 1, dimension):
                sum_X += upper[i][k] * X_res[k][j]
            X_res[i][j] = (1 / upper[i][i]) * (Y_res[i][j] - sum_X)
    

    return X_res

#Calculating the product of a matrix and vector 
def matrix_product_with_matrix_and_vector(matrix, vector, dimension):
    mat = [[0] * dimension for _ in range(dimension)]
    res = [0] * dimension
    for i in range(dimension):
        for j in range(dimension):
            mat[j][i] = matrix[j][i] * vector[i]
    
    for k in range(dimension):
        row_sum = 0
        for l in range(dimension):
            row_sum += mat[k][l]
        res[k] = row_sum
    
    return res

def scalar_product_from_transpose_and_vector(v1, v2, dimension):
    res = 0
    for i in range(dimension):
        res += v1[i] * v2[i]
    return res

#Regularising a matrix to help prevent division by zero error
def regularize(matrix, dimension, epsilon=1e-6):
    for i in range(dimension):
        matrix[i][i] += epsilon
    return matrix

def multivariate_gaussian(mean, covariance_matrix, feature_vector, dimension):
    resultant_vector = calculate_vector(feature_vector, mean)
    cov_copy = [row[:] for row in covariance_matrix]  # make a copy!
    cov_copy = regularize(cov_copy, dimension)
    lower_triangular, upper_triangular = LU_decomposition(cov_copy, dimension)
    inverse = matrix_inverse(lower_triangular, upper_triangular, dimension)
    vector_prod = matrix_product_with_matrix_and_vector(inverse, resultant_vector, dimension)
    scalar_prod = scalar_product_from_transpose_and_vector(resultant_vector, vector_prod, dimension)
    determinant = calculate_determinant(upper_triangular, dimension)
    exponential_term = math.exp(-0.5 * scalar_prod)
    denominator = 1/(((2 * math.pi)**(dimension/2)) * (abs(determinant) ** 0.5))
    res = denominator * exponential_term
    return res

In [16]:
covar_mat = calculate_covariance_matrix(dimension[1])
def test_accuracy(dimension, x_test_len):
    correct = 0
    for i in range(x_test_len):
        pred = 0
        prediction_zero = multivariate_gaussian(mu_zero_arr, covar_mat, x_test[i].tolist(), dimension) * (1-phi)
        prediction_one = multivariate_gaussian(mu_one_arr, covar_mat, x_test[i].tolist(), dimension) * phi
        if(prediction_one > prediction_zero):
            pred = 1

        if(pred == 1 and y_test[i] == 'M'):
            correct += 1
        elif(pred == 0 and y_test[i] != 'M'):
            correct += 1
        
        res = 'M'
        if pred == 0:
            res = 'B'
            
        print(res)
        print(y_test[i])
        print("")
    
    return (correct/len(x_test)) * 100


print(test_accuracy(dimension[1], len(x_test)))

M
M

B
B

B
B

B
B

M
M

B
B

B
B

M
M

M
M

B
B

M
M

M
M

B
B

M
M

M
M

B
B

M
M

B
B

B
B

M
M

M
M

B
B

B
B

M
M

B
B

B
B

B
B

B
B

B
B

B
B

B
B

M
M

B
B

M
M

M
M

B
B

B
B

M
M

B
B

M
M

B
B

M
M

B
B

B
B

B
B

B
B

B
B

M
M

B
B

B
B

B
M

B
B

B
B

B
B

B
B

B
B

M
M

B
B

B
B

M
M

B
B

B
B

M
M

B
B

B
B

B
B

M
M

B
B

B
B

B
B

B
B

B
B

B
B

B
B

M
M

B
B

B
B

B
B

B
B

B
B

B
B

B
B

M
M

B
B

B
B

B
B

M
M

M
M

M
M

M
M

B
B

M
M

B
B

M
M

B
B

M
M

B
B

B
B

B
B

B
B

M
M

M
M

M
M

B
M

M
M

B
B

M
M

B
B

M
M

M
M

B
B

B
M

B
B

B
M

96.49122807017544
